In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/test_predicted_fingerprints.npy
/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


# 1. Importaciones y Configuración

In [4]:
import os

# Carpeta contenedora donde está alojado el dataset en tu Input
WHEEL_DIR = '/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/'

# Instalación buscando la rueda compatible en esa carpeta
!pip install --no-index --find-links={WHEEL_DIR} rdkit --quiet

# Verificación
import rdkit
from rdkit import Chem
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc
from rdkit.Chem import rdFingerprintGenerator

print(f"✓ RDKit v{rdkit.__version__} instalado con éxito en modo offline.")

# Rutas de entrada
DATA_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TEST_PATH = os.path.join(DATA_DIR, 'test.parquet')

# Ruta de tus predicciones de la Fase 2 (ajusta el nombre del dataset si difiere)
PHASE2_DIR = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints'
PRED_FP_PATH = os.path.join(PHASE2_DIR, 'test_predicted_fingerprints.npy')

print("Rutas configuradas correctamente.")

✓ RDKit v2025.03.6 instalado con éxito en modo offline.
Rutas configuradas correctamente.


# 2. Carga test y predicciones fingerprints

In [5]:
print("Cargando test.parquet y fingerprints predichos...")
test_df = pd.read_parquet(TEST_PATH)
test_fps = np.load(PRED_FP_PATH)

print(f"Total de espectros en Test: {len(test_df):,}")
print(f"Shape de Fingerprints predichos: {test_fps.shape}")

# Agrupar espectros por 'molecule_id' promediando los fingerprints si una molécula tiene varios espectros
unique_molecules = test_df[['molecule_id']].drop_duplicates().reset_index(drop=True)
print(f"Moléculas únicas a predecir: {len(unique_molecules):,}")

# Promedio de fingerprints por molecule_id
test_df['fp_idx'] = np.arange(len(test_df))
mol_to_fp = {}

for mol_id, group in test_df.groupby('molecule_id'):
    indices = group['fp_idx'].values
    avg_fp = test_fps[indices].mean(axis=0)
    # Binarizar el fingerprint promediado con umbral 0.5
    binary_fp = (avg_fp > 0.5).astype(np.uint8)
    mol_to_fp[mol_id] = binary_fp

print("Agregación de fingerprints por molecule_id completada.")

Cargando test.parquet y fingerprints predichos...
Total de espectros en Test: 1,213
Shape de Fingerprints predichos: (1213, 1024)
Moléculas únicas a predecir: 400
Agregación de fingerprints por molecule_id completada.


# 3. Búsqueda y Binarización de Candidatos (Librería Local / PubChem Subconjunto)

In [6]:
print("Cargando base de datos de candidatos desde train.parquet...")
TRAIN_PATH = os.path.join(DATA_DIR, 'train.parquet')

# Leemos únicamente las columnas necesarias para no saturar la RAM
train_df = pd.read_parquet(TRAIN_PATH, columns=['normalized_smiles', 'inchikey14'])

# Candidatos únicos por InChIKey14
candidate_df = train_df.drop_duplicates(subset=['inchikey14']).reset_index(drop=True)

# Liberar memoria del DataFrame pesado de entrenamiento
del train_df
gc.collect()

print(f"Base de candidatos únicos a procesar: {len(candidate_df):,}")

# Instanciar el generador moderno de Morgan Fingerprints de RDKit (ECFP4, 1024 bits)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

# Pre-asignar matriz NumPy binaria compacta (1024 bits para coincidir con la Fase 2)
num_candidates = len(candidate_df)
candidate_fps_matrix = np.zeros((num_candidates, 1024), dtype=np.uint8)
valid_smiles = []

print("Calculando Morgan Fingerprints con MorganGenerator (sin warnings)...")
valid_count = 0

for idx, row in tqdm(candidate_df.iterrows(), total=num_candidates):
    smi = row['normalized_smiles']
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        # Usar la API moderna GetFingerprint
        fp = morgan_gen.GetFingerprint(mol)
        candidate_fps_matrix[valid_count] = np.frombuffer(fp.ToBitString().encode('utf-8'), 'u1') - ord('0')
        valid_smiles.append(smi)
        valid_count += 1

# Recortar la matriz al número exacto de SMILES válidos
candidate_fps_matrix = candidate_fps_matrix[:valid_count]
candidate_df = pd.DataFrame({'normalized_smiles': valid_smiles})

print(f"✓ Matriz de candidatos lista: {candidate_fps_matrix.shape} | RAM ocupada: {candidate_fps_matrix.nbytes / (1024**2):.2f} MB")

Cargando base de datos de candidatos desde train.parquet...
Base de candidatos únicos a procesar: 275,810
Calculando Morgan Fingerprints con MorganGenerator (sin warnings)...


100%|██████████| 275810/275810 [01:52<00:00, 2452.10it/s]


✓ Matriz de candidatos lista: (275810, 1024) | RAM ocupada: 269.35 MB


# 4. Recuperación de Top-25 Candidatos por Similitud Tanimoto

In [7]:
print("Calculando similitud Tanimoto matricial para las moléculas de Test...")

def batch_tanimoto_similarity(query_fp, candidate_matrix):
    """
    Calcula la similitud Tanimoto vectorizada:
    Tanimoto(A, B) = (A . B) / (|A| + |B| - A . B)
    """
    cand_sums = candidate_matrix.sum(axis=1)
    query_sum = query_fp.sum()
    
    # Producto escalar (bits coincidentes en 1)
    intersection = np.dot(candidate_matrix, query_fp)
    
    # Denominador de la fórmula Tanimoto
    union = cand_sums + query_sum - intersection
    
    # Cálculo seguro evitando división por cero
    scores = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)
    return scores

results = []
top_k = 25

for mol_id in tqdm(unique_molecules['molecule_id']):
    pred_bit_array = mol_to_fp[mol_id]
    
    # Similitud vectorial en NumPy
    scores = batch_tanimoto_similarity(pred_bit_array, candidate_fps_matrix)
    
    # Extraer los índices del Top-25 con mayor puntuación
    top_indices = np.argpartition(scores, -top_k)[-top_k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    
    top_smiles = candidate_df.iloc[top_indices]['normalized_smiles'].tolist()
    
    # Unir los candidatos por punto y coma (;) como exige el formato oficial
    results.append({
        'molecule_id': mol_id,
        'normalized_smiles': ";".join(top_smiles)
    })

# Formatear el DataFrame final de candidatos recuperados
retrieved_df = pd.DataFrame(results)

print("\nMuestra de candidatos recuperados para las primeras moléculas:")
print(retrieved_df.head())

Calculando similitud Tanimoto matricial para las moléculas de Test...


100%|██████████| 400/400 [02:37<00:00,  2.54it/s]


Muestra de candidatos recuperados para las primeras moléculas:
  molecule_id                                  normalized_smiles
0    m_005e53  CC(=O)Nc1ccc(Br)cc1;CNC(=O)CCN1CCN(c2cccc(Br)c...
1    m_006153  CCn1nc(C2CC2)cc1C(=O)O;O=C(O)c1cc(C(=O)O)c(C(=...
2    m_00b5aa  CCn1nccc1C(=O)O;Cn1nccc1C(=O)O;O=C(O)c1ccnn1CC...
3    m_01e922  CC(=O)c1ccccc1C(C)=O;O=C(O)c1cc(C(=O)O)c(C(=O)...
4    m_0259d4  O=C(CC(=O)c1ccccc1)c1ccccc1;CC(=O)Nc1ccccc1;CO...


# 5. Verificación de Formato y Guardado de Candidatos

In [10]:
# 1. Asegurar el nombre de columna 'smiles' requerida por la competición
if 'normalized_smiles' in retrieved_df.columns:
    retrieved_df = retrieved_df.rename(columns={'normalized_smiles': 'smiles'})

retrieved_df = retrieved_df[['molecule_id', 'smiles']]

# 2. Assertions de integridad
assert len(retrieved_df) == len(unique_molecules), "Error: El número de filas no coincide con el test set."
assert not retrieved_df.isnull().values.any(), "Error: Existen valores nulos."
assert retrieved_df['molecule_id'].nunique() == len(retrieved_df), "Error: Hay IDs duplicados."

# Verificar candidatos por molécula
max_candidates = retrieved_df['smiles'].apply(lambda x: len(x.split(';'))).max()
print(f"✓ Candidatos máximos por fila: {max_candidates} (Límite permitido: 25)")

retrieved_df.head()

# 3. Guardar archivo final del Notebook 3
OUTPUT_FILE = 'submission.csv'
retrieved_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✓ ¡Notebook 3 procesado exitosamente! Archivo guardado como: {OUTPUT_FILE}")

✓ Candidatos máximos por fila: 25 (Límite permitido: 25)

✓ ¡Notebook 3 procesado exitosamente! Archivo guardado como: submission.csv
